# 03 — Resultados Presidencial

Top partidos a nivel nacional + desglose por región y urbano/rural (exterior vs Perú). Usamos `actas_votos_tidy` que ya tiene el contexto geográfico joinado.

In [ ]:
import polars as pl

tidy = pl.scan_parquet('parquet/actas_votos_tidy.parquet')
print('schema:', tidy.collect_schema().names())

In [ ]:
# Top 10 partidos a nivel nacional en Presidencial
top_nacional = (
    tidy.filter(pl.col('idEleccion') == 10)
    .filter(~pl.col('es_especial'))
    .group_by('partido')
    .agg(pl.col('nvotos').sum().alias('nvotos'))
    .sort('nvotos', descending=True)
    .head(10)
    .collect()
)
total_validos = top_nacional['nvotos'].sum()
top_nacional.with_columns(
    (pl.col('nvotos') * 100.0 / total_validos).round(2).alias('pct_top10')
)

In [ ]:
# Votos especiales (blancos, nulos, impugnados) a nivel nacional
especiales = (
    tidy.filter(pl.col('idEleccion') == 10)
    .filter(pl.col('es_especial'))
    .group_by('partido')
    .agg(pl.col('nvotos').sum().alias('nvotos'))
    .sort('nvotos', descending=True)
    .collect()
)
especiales

In [ ]:
# Top partido por cada distrito electoral
por_de = (
    tidy.filter(pl.col('idEleccion') == 10)
    .filter(~pl.col('es_especial'))
    .group_by(['idDistritoElectoral', 'partido'])
    .agg(pl.col('nvotos').sum().alias('nvotos'))
    .collect()
)
ganador_por_de = (
    por_de.sort(['idDistritoElectoral', 'nvotos'], descending=[False, True])
    .group_by('idDistritoElectoral', maintain_order=True)
    .head(1)
    .sort('idDistritoElectoral')
)
ganador_por_de

In [ ]:
# Comparar top 5 Perú vs Exterior
def top5_por_ambito(ambito: int):
    return (
        tidy.filter(pl.col('idEleccion') == 10)
        .filter(~pl.col('es_especial'))
        .filter(pl.col('idAmbitoGeografico') == ambito)
        .group_by('partido')
        .agg(pl.col('nvotos').sum().alias('nvotos'))
        .sort('nvotos', descending=True)
        .head(5)
        .collect()
    )

print('=== Perú ===')
print(top5_por_ambito(1))
print('\n=== Exterior ===')
print(top5_por_ambito(2))